# Aerial and latitude–height animations (Eriswil)

Notebook aligned with the publication pipeline: `scripts/processing_chain/run_publication_figures.py` reads
`config/publication_figures.yaml` (ensemble, paths, per-job CLI defaults). Process-budget and time-zero context live in
`config/psd_process_evolution.yaml`.

**Outputs:** frames under `output/gfx/png/aerials/<cs_run>/`, MP4/GIF under `output/gfx/mp4/` (same tree as other promoted figures; `run_publication_figures.py` syncs `output/gfx` → `output/gallery`).

**Data:** the notebook resolves `.../ensemble_output/<cs_run>` using the same fallbacks as `get_output_root` in `processing_paths.py`: `paths.server_root` in YAML, then `CS_RUNS_DIR`, `POLARCAP_OUTPUT_ROOT`, the Levante default tree, then local `scripts/data/registry/processed`. Jupyter kernels often lack shell exports—set `CS_RUNS_DIR` in IPython startup or assign `os.environ["CS_RUNS_DIR"] = "..."` before the config cell.

In [1]:
import glob
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as md
import numpy as np
import pandas as pd
import xarray as xr
import yaml
from matplotlib.colors import LogNorm
from dask.diagnostics import ProgressBar


def find_repo_root(start: Path | None = None) -> Path:
    """Match notebooks/spectral_waterfall_minimal: stable root from any cwd."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "utilities" / "spectral_waterfall.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook inside the polarcap_analysis repository.")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
for path in (REPO_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from utilities.model_helpers import fetch_3d_data, convert_units_3d
from utilities.namelist_metadata import update_dataset_metadata
from utilities.plotting import new_fjet2, set_name_tick_params, add_ruler
from utilities.processing_paths import find_ensemble_output_for_cs_run
from utilities.style_profiles import apply_publication_style, proc_color
import utilities.tools as tools

xr.set_options(keep_attrs=True)
apply_publication_style()

PUBLICATION_YAML = REPO_ROOT / "config" / "publication_figures.yaml"
PSD_YAML = REPO_ROOT / "config" / "psd_process_evolution.yaml"


def _job_tokens(pub: dict, job_key: str) -> list[str]:
    out: list[str] = []
    for x in (pub.get("default_args") or {}).get(job_key, []) or []:
        out.extend(str(x).split())
    return out


def _flag_value(tokens: list[str], name: str) -> str | None:
    try:
        i = tokens.index(name)
    except ValueError:
        return None
    return tokens[i + 1] if i + 1 < len(tokens) else None


In [2]:
# Load publication + PSD configs (same files as the publication runner / cloud overview script)
with open(PUBLICATION_YAML) as f:
    pub = yaml.safe_load(f) or {}
with open(PSD_YAML) as f:
    psd = yaml.safe_load(f) or {}

ens = pub.get("ensemble") or {}
paths_cfg = pub.get("paths") or {}
cs_run = str(ens.get("cs_run", "cs-eriswil__20260318_153631"))
domain_xy = str(ens.get("domain", "200x160"))
flare_idx = int((psd.get("selection") or {}).get("experiment_index_default", 0))

server_root = (paths_cfg.get("server_root") or "").strip() or None
ens_out, _tried = find_ensemble_output_for_cs_run(cs_run, config_runs_root=server_root)
if not ens_out:
    raise FileNotFoundError(
        "Could not find ensemble_output/"
        + cs_run
        + ". Tried:\n  "
        + "\n  ".join(_tried)
        + "\nSet paths.server_root in publication_figures.yaml, or export CS_RUNS_DIR "
        "(parent of RUN_ERISWILL_*x100), or POLARCAP_OUTPUT_ROOT (runs root or ensemble_output). "
        "Jupyter often does not inherit shell env: set variables in ~/.ipython/profile_default/startup "
        "or in this cell via os.environ before re-run."
    )
model_data_path = Path(ens_out) / cs_run

cf_tok = _job_tokens(pub, "cloud_field_overview")
extpar_high = _flag_value(cf_tok, "--extpar-high")
extpar_low = _flag_value(cf_tok, "--extpar-low")
plot_start_s = _flag_value(cf_tok, "--plot-start")
plot_end_s = _flag_value(cf_tok, "--plot-end")
if domain_xy == "200x160" and extpar_high:
    extpar_file = Path(extpar_high).expanduser()
elif extpar_low:
    extpar_file = Path(extpar_low).expanduser()
else:
    extpar_file = Path(extpar_high or extpar_low or "").expanduser()

seed_start = np.datetime64((psd.get("time") or {}).get("seed_start", "2023-01-25T12:29:50"))

GFX_PNG = REPO_ROOT / "output" / "gfx" / "png" / "aerials" / cs_run.replace("/", "_")
GFX_MP4 = REPO_ROOT / "output" / "gfx" / "mp4"
GFX_PNG.mkdir(parents=True, exist_ok=True)
GFX_MP4.mkdir(parents=True, exist_ok=True)

print("cs_run:", cs_run, "domain:", domain_xy)
print("model_data_path:", model_data_path)
print("extpar_file:", extpar_file)
print("plot window (cloud_field_overview preset):", plot_start_s, "..", plot_end_s)


FileNotFoundError: Could not find ensemble_output/cs-eriswil__20260318_153631. Tried:
  /Users/schimmel/data/cosmo-specs/meteograms
  /work/bb1262/user/schimmel/cosmo-specs-torch/cosmo-specs-runs
  /Users/schimmel/Library/Mobile Documents/com~apple~CloudDocs/code/polarcap/python/polarcap_analysis/scripts/data/registry/processed
Set paths.server_root in publication_figures.yaml, or export CS_RUNS_DIR (parent of RUN_ERISWILL_*x100), or POLARCAP_OUTPUT_ROOT (runs root or ensemble_output). Jupyter often does not inherit shell env: set variables in ~/.ipython/profile_default/startup or in this cell via os.environ before re-run.

## Load 3D flare member

In [ ]:
flist_3d = sorted(glob.glob(str(model_data_path / "3D_??????????????.nc")))
if not flist_3d:
    raise FileNotFoundError(f"No 3D_*.nc under {model_data_path}")

json_files = sorted(model_data_path.glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No ensemble json next to 3D files in {model_data_path}")
with open(json_files[0]) as jsonfile:
    meta = json.load(jsonfile)

exp_names = [Path(f).name.split("_")[-1].split(".")[0] for f in flist_3d]
flare_candidates = [exp for exp in exp_names if meta[exp]["INPUT_ORG"]["sbm_par"]["lflare"]]
flare_exp_name = flare_candidates[flare_idx]
flare_exp_nc_file = next(f for f in flist_3d if flare_exp_name in f)
print("flare member:", flare_exp_name)

ds_3d = fetch_3d_data(
    str(flare_exp_nc_file),
    str(extpar_file),
    meta[flare_exp_name]["INPUT_ORG"],
    var_sets=["meteo", "bulk", "spec"],
    chunks={"time": 1},
)
ds_3d = update_dataset_metadata(ds_3d)
ds_3d = ds_3d.isel(altitude=slice(80, None))
ds_3d = convert_units_3d(ds_3d, ds_3d["rho"])


In [ ]:
# Map-view layout: figure size from publication DPI (matches savefig in analysis scripts)
_dpi = float(plt.rcParams["savefig.dpi"])
pixel_w, pixel_h = 1920, 1080
resolution_label = "400m" if domain_xy == "50x40" else "100m"
resolution_deg = 0.004 if domain_xy == "50x40" else 0.001

cfg = {
    "resolution": resolution_label,
    "resolution_deg": resolution_deg,
    "dpi": int(_dpi),
    "pixel_size": (pixel_w, pixel_h),
    "poolsize": min(8, os.cpu_count() or 4),
    "plot_all_frames": True,
    "flare_lat": 47.07425,
    "flare_lon": 7.90522,
    "origin_lat": 47.070522,
    "origin_lon": 7.872991,
    "plot_xlim": (7.7671843, 7.94),
    "plot_ylim": (47.02, 47.12),
    "delta_x": float(1e3 * np.mean(np.diff(ds_3d.longitude.values)) * 111.13295254925466),
    "delta_y": float(1e3 * np.mean(np.diff(ds_3d.latitude.values)) * 111.13295254925466),
    "delta_t": float(np.mean(np.diff(ds_3d.time.astype("datetime64[s]")).astype(float))),
    "n_lon": ds_3d.longitude.size,
    "n_lat": ds_3d.latitude.size,
    "n_time": ds_3d.time.size,
    "vel_lims": [-0.5, 0.5],
    "v_lims_qi": [1e-4, 1e0],
    "tick_size": int(plt.rcParams["xtick.labelsize"] + 1),
    "axis_size": int(plt.rcParams["axes.labelsize"] + 2),
    "timer_size": int(plt.rcParams["figure.titlesize"] + 6),
}

data_extpar = xr.open_mfdataset(str(extpar_file), chunks="auto")
lat2d = data_extpar["lat"].values[7:-7, 7:-7]
lon2d = data_extpar["lon"].values[7:-7, 7:-7]
height = data_extpar["HSURF"].values[7:-7, 7:-7]


In [ ]:
# Column ice water path (qi + qs) for plan view
mod = ds_3d[["qi", "qs", "dz"]].sel(
    latitude=slice(None, cfg["flare_lat"] + 2.0 * cfg["resolution_deg"]),
    longitude=slice(None, cfg["flare_lon"] + 2.0 * cfg["resolution_deg"]),
)
mod_latlon = (mod[["qi", "qs"]] * mod["dz"]).sum("altitude")
with ProgressBar():
    mod_latlon = xr.where(mod_latlon < 1e-7, np.nan, mod_latlon).persist()
IWC = (mod_latlon["qi"] + mod_latlon["qs"]).values


In [ ]:
# Tracking overlays (optional): time slice from publication cloud_field_overview preset
if plot_start_s and plot_end_s:
    plot_time_frame = [np.datetime64(plot_start_s), np.datetime64(plot_end_s)]
else:
    plot_time_frame = [seed_start, seed_start + np.timedelta64(2, "h")]

tracks_csv = tools.load_tracking_csv(str(model_data_path / f"{flare_exp_name}_qi+qs_tobac_track.csv"))
tracks_csv_qi = tools.load_tracking_csv(str(model_data_path / f"{flare_exp_name}_qi_tobac_track.csv"))
tracks_csv_qs = tools.load_tracking_csv(str(model_data_path / f"{flare_exp_name}_qs_tobac_track.csv"))


def _tracks_by_cell(df, t0, step=6):
    out = []
    for _, g in df.groupby("cell"):
        sub = g[pd.to_datetime(g["time"]) >= t0][::step]
        if len(sub):
            out.append(sub)
    return out


cell_tracks = _tracks_by_cell(tracks_csv, plot_time_frame[0])
cell_tracks_qi = _tracks_by_cell(tracks_csv_qi, plot_time_frame[0])
cell_tracks_qs = _tracks_by_cell(tracks_csv_qs, plot_time_frame[0])


## Plotting helpers (plan view)

In [ ]:
def add_map_annotations(ax, cfg):
    ax.scatter(cfg["origin_lon"], cfg["origin_lat"], s=70, marker="x", color="red", zorder=2)
    ax.scatter(
        cfg["flare_lon"],
        cfg["flare_lat"],
        s=50,
        marker="o",
        facecolor="none",
        edgecolor="white",
        lw=2.5,
        zorder=2,
    )
    ax.scatter(
        cfg["flare_lon"],
        cfg["flare_lat"],
        s=50,
        marker="o",
        facecolor="none",
        edgecolor="red",
        lw=1.0,
        zorder=2,
    )
    add_ruler(ax, 47.05, 7.804, cfg["flare_lat"], cfg["flare_lon"])
    seeding = {
        "royalblue": ([7.90476, 7.90568], [47.07602, 47.07248]),
        "orange": ([7.89828, 7.89919], [47.07526, 47.07172]),
        "green": ([7.91125, 7.91216], [47.07676, 47.07322]),
    }
    for _c, (lon, lat) in seeding.items():
        dx, dy = np.diff(lon)[0], np.diff(lat)[0]
        ext_lon, ext_lat = [lon[0] - dx, lon[1] + dx], [lat[0] - dy, lat[1] + dy]
        ax.plot(lon, lat, color="black", lw=0.9)
        ax.plot(ext_lon, ext_lat, color="black", alpha=0.4, lw=0.6)
        ax.scatter(lon, lat, color="black", marker=".", s=15)


def setup_plot_lat_lon(cfg):
    w_in = cfg["pixel_size"][0] / cfg["dpi"]
    h_in = cfg["pixel_size"][1] / cfg["dpi"]
    fig, ax = plt.subplots(figsize=(w_in, h_in), constrained_layout=True)
    ax.set_xlim(cfg["plot_xlim"])
    ax.set_ylim(cfg["plot_ylim"])
    set_name_tick_params(ax)
    ax.tick_params(axis="both", which="major", labelsize=cfg["tick_size"])
    return fig, ax


def add_plan_colorbars(fig, ax, pmesh, cfg):
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    cb_qi = fig.colorbar(pmesh[0], ax=ax, extend="both", shrink=0.7, aspect=15, pad=0.01)
    cb_qi.ax.text(
        0.52,
        4e-1 * cfg["v_lims_qi"][0],
        "IWP\n(g/m²)",
        ha="center",
        va="top",
        fontsize=cfg["axis_size"] - 1,
    )
    cb_surf = fig.colorbar(
        pmesh[1], ax=ax, extend="both", shrink=0.5, aspect=20, pad=0.1, orientation="horizontal"
    )
    cb_surf.set_label("height a.m.s.l. / (m)", fontsize=cfg["tick_size"], labelpad=2)
    cb_surf.ax.tick_params(labelsize=cfg["tick_size"] - 1)
    return [cb_qi, cb_surf]


def update_plan_frame(iframe, fig, ax, pmesh, cell_plots, cfg, time, z_val, cell_tracks_1, cell_tracks_2):
    for txt in fig.texts:
        if "UTC" in txt.get_text():
            txt.set_visible(False)
    pmesh[0].set_array(z_val[iframe, :, :].ravel())
    ts = np.datetime_as_string(time[iframe], unit="s").replace("T", "  ")
    fig.text(
        0.96,
        0.99,
        f"{ts[-8:]} UTC",
        ha="right",
        va="bottom",
        fontweight="semibold",
        fontsize=cfg["timer_size"],
    )
    if cell_tracks_1 or cell_tracks_2:
        for p in cell_plots:
            p.remove()
        cell_plots.clear()
    cur = time[iframe]
    if cell_tracks_2 is not None:
        for ct in cell_tracks_2:
            tr = ct[pd.to_datetime(ct["time"]) <= cur]
            if not tr.empty:
                cell_plots.append(
                    ax.scatter(tr.longitude.values, tr.latitude.values, c="#F18F01", s=30, alpha=0.95, marker="+", zorder=90)
                )
    if cell_tracks_1 is not None:
        for ct in cell_tracks_1:
            tr = ct[pd.to_datetime(ct["time"]) <= cur]
            if not tr.empty:
                cell_plots.append(
                    ax.scatter(tr.longitude.values, tr.latitude.values, c="#2E86AB", s=30, alpha=0.95, marker="x", zorder=90)
                )
    out = GFX_PNG / f"lat_lon_frame_{iframe:03d}_{cfg['resolution']}_area.png"
    fig.savefig(out, dpi=cfg["dpi"], bbox_inches="tight")


## Plan-view animation

In [ ]:
png_path = str(GFX_PNG)
os.makedirs(png_path, exist_ok=True)
fig, ax = setup_plot_lat_lon(cfg)
add_map_annotations(ax, cfg)
pm_surf = ax.pcolormesh(lon2d, lat2d, height, cmap="terrain", vmin=300, alpha=0.6, zorder=1)
pm_qi = ax.pcolormesh(
    mod_latlon.longitude2D,
    mod_latlon.latitude2D,
    IWC[0, :, :],
    cmap=new_fjet2,
    norm=LogNorm(*cfg["v_lims_qi"]),
    zorder=50,
)
pmesh = [pm_qi, pm_surf]
cell_plots = []
fig.text(0.5, 0.18, "longitude / (°)", ha="center", va="center", fontsize=cfg["axis_size"])
fig.text(
    -0.01, 0.6, "latitude / (°)", ha="center", va="center", fontsize=cfg["axis_size"], rotation="vertical"
)
add_plan_colorbars(fig, ax, pmesh, cfg)

for pattern in ("*.png",):
    for f in glob.glob(os.path.join(png_path, pattern)):
        os.remove(f)
print("Cleaned:", png_path)

if cfg["plot_all_frames"]:
    for iframe in range(IWC.shape[0]):
        update_plan_frame(
            iframe, fig, ax, pmesh, cell_plots, cfg, mod_latlon.time.values, IWC, cell_tracks_qi, cell_tracks_qs
        )
    print("All plan-view frames saved")

input_pattern = os.path.join(png_path, f"lat_lon_frame_%03d_{cfg['resolution']}_area.png")
out_mp4 = GFX_MP4 / f"lat_lon_{flare_exp_name}_{cfg['resolution']}_{cs_run.replace('__', '_')}.mp4"
out_gif = out_mp4.with_suffix(".gif")
tools.convert_to_video(str(input_pattern), str(out_mp4), resolution="1920:1080", loop_count=2, framerate=15)
tools.convert_to_gif(str(input_pattern), str(out_gif), scale_factor=0.5, fps=20)
print(out_mp4)
print(out_gif)


## Latitude–height section + domain ice mass

Uses the same `cfg`, `ds_3d`, and `GFX_*` roots. Time-series colours use `proc_color` (Okabe–Ito) from `style_profiles`.

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import AutoMinorLocator


def compute_integrated_water_contents(ds_3d_local):
    dss = ds_3d_local[["qc", "qi", "qs", "qv", "dz"]]
    wc = {
        "IWC": (dss["qi"] * dss["dz"] * cfg["delta_x"] * cfg["delta_y"]).sum(
            ["altitude", "longitude", "latitude"]
        ),
        "LWC": (dss["qc"] * dss["dz"] * cfg["delta_x"] * cfg["delta_y"]).sum(
            ["altitude", "longitude", "latitude"]
        ),
        "SWC": (dss["qs"] * dss["dz"] * cfg["delta_x"] * cfg["delta_y"]).sum(
            ["altitude", "longitude", "latitude"]
        ),
        "VC": (dss["qv"] * dss["dz"] * cfg["delta_x"] * cfg["delta_y"]).sum(
            ["altitude", "longitude", "latitude"]
        ),
    }
    wc["TWC"] = wc["IWC"] + wc["LWC"] + wc["VC"]
    wc["TIWC"] = wc["IWC"] + wc["SWC"]
    return xr.Dataset(wc)


def setup_plot_with_timeseries(cfg):
    total_width = cfg["pixel_size"][0] / cfg["dpi"]
    main_height = cfg["pixel_size"][1] / cfg["dpi"]
    ts_height = main_height * 0.33
    fig = plt.figure(figsize=(total_width, main_height + ts_height), constrained_layout=True)
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 3], hspace=0.05)
    ax_ts = fig.add_subplot(gs[0])
    ax_main = fig.add_subplot(gs[1])
    ax_main.set_ylim((600, 1400))
    set_name_tick_params(ax_main)
    set_name_tick_params(ax_ts)
    ax_hist = inset_axes(
        ax_main,
        width="30%",
        height="40%",
        loc="upper right",
        bbox_to_anchor=(-0.05, -0.05, 1.0, 1.0),
        bbox_transform=ax_main.transAxes,
    )
    ax_hist.tick_params(axis="both", which="major", labelsize=cfg["tick_size"] - 2)
    ax_hist.patch.set_alpha(0.8)
    return fig, ax_main, ax_ts


def plot_timeseries_dual_axis(ax_ts, water_contents, current_time, cfg):
    ax_ts.clear()
    time = water_contents["IWC"].time.values
    colors_left = {"IWC": proc_color("CONDENSATION"), "SWC": proc_color("DROP_COLLISION"), "TIWC": "#000000"}
    lines, labels = [], ["IWC", "SWC", "TIWC"]
    for key in labels:
        line, = ax_ts.plot(
            time,
            water_contents[key].values * 1e-3,
            color=colors_left[key],
            lw=2.0 if key == "TIWC" else 1.5,
            alpha=1.0 if key == "TIWC" else 0.9,
            label=key,
        )
        lines.append(line)
    ax_ts.axvline(current_time, color="black", linestyle="--", lw=1, alpha=0.75, zorder=10)
    ax_ts.set_xlim(time[0], time[-1])
    ax_ts.set_xlabel("time / (UTC)", fontsize=cfg["axis_size"], color="black")
    ax_ts.set_ylabel("ice mass / (kg)", fontsize=cfg["axis_size"], color="black")
    ax_ts.set_axisbelow(True)
    ax_ts.ticklabel_format(axis="y", style="scientific", scilimits=(-3, +3))
    ax_ts.xaxis.set_major_formatter(md.DateFormatter("%H:%M"))
    ax_ts.xaxis.set_minor_locator(AutoMinorLocator())
    ax_ts.yaxis.set_minor_locator(AutoMinorLocator())
    ax_ts.tick_params(axis="both", which="major", direction="inout", top=True, right=True, labelsize=cfg["tick_size"])
    ax_ts.tick_params(axis="both", which="minor", direction="inout", top=True, right=True, labelsize=cfg["tick_size"])
    ax_ts.grid(which="both", linestyle="--", alpha=0.15, lw=0.5, color="black")
    ax_ts.grid(which="minor", linestyle="--", alpha=0.05, lw=0.25, color="black")
    ax_ts.legend(lines, labels, loc="upper left", ncol=4, fontsize=cfg["tick_size"], framealpha=0.9, edgecolor="none")


def update_frame_enhanced(iframe, fig, ax_main, ax_ts, pmesh, cfg, time_values, IWC_lat, water_contents):
    for txt in fig.texts:
        if "UTC" in txt.get_text():
            txt.set_visible(False)
    pmesh[0].set_array(IWC_lat[iframe, :, :].ravel())
    cur = time_values[iframe]
    plot_timeseries_dual_axis(ax_ts, water_contents, cur, cfg)
    ts = np.datetime_as_string(cur, unit="s").replace("T", "  ")
    fig.text(
        0.96,
        0.99,
        f"{ts[-8:]} UTC",
        ha="right",
        va="top",
        fontweight="semibold",
        fontsize=cfg["timer_size"],
    )
    out = GFX_PNG / f"lat_height_frame_{iframe:03d}_{cfg['resolution']}.png"
    fig.savefig(out, dpi=cfg["dpi"], bbox_inches="tight")


water_contents = compute_integrated_water_contents(ds_3d)
with ProgressBar():
    water_contents = water_contents.persist()

data_extpar2 = xr.open_mfdataset(str(extpar_file), chunks="auto")
lat1d = data_extpar2["lat"].mean("rlon").values[7:-7]
surf_height = data_extpar2["HSURF"].values[7:-7, 7:-7]
surf_height_mean = data_extpar2["HSURF"].mean("rlon").values[7:-7]
mod_lat = ds_3d[["qi", "qs", "qc", "qv"]].sel(
    latitude=slice(None, cfg["flare_lat"] + 2.0 * cfg["resolution_deg"]),
    longitude=slice(None, cfg["flare_lon"] + 2.0 * cfg["resolution_deg"]),
)
mod_lat = xr.where(mod_lat < 1e-6, np.nan, mod_lat)
mod_lat = (mod_lat * cfg["delta_x"]).sum("longitude")
with ProgressBar():
    mod_lat = mod_lat.persist()
height1d = mod_lat.altitude.values
IWC_lat = (mod_lat["qi"] + mod_lat["qs"]).values
print("lat-height array", IWC_lat.shape)


In [ ]:
idx_time = min(130, IWC_lat.shape[0] - 1)
fig, ax_main, ax_ts = setup_plot_with_timeseries(cfg)
pm_surf = ax_main.plot(lat1d, surf_height_mean, color="black", alpha=0.9, lw=2)
pml_surf = ax_main.plot(
    lat1d, surf_height, color="black", alpha=0.035 if "100m" in cfg["resolution"] else 0.075
)
pm_qi = ax_main.pcolormesh(
    lat1d[: IWC_lat.shape[-1]],
    height1d,
    IWC_lat[idx_time, :, :],
    cmap=new_fjet2,
    norm=LogNorm(*cfg["v_lims_qi"]),
    zorder=2,
)
pmesh_lh = [pm_qi, pm_surf]
fig.text(0.45, -0.02, "latitude / (°)", ha="center", va="center", fontsize=cfg["axis_size"])
fig.text(
    0.0, 0.4, "height a.m.s.l. / (m)", ha="center", va="center", fontsize=cfg["axis_size"], rotation="vertical"
)
cb_qi = fig.colorbar(
    pm_qi, ax=ax_main, pad=0.025, extend="both", shrink=0.8, aspect=20, label="ice water path / (g/m²)"
)
cb_qi.ax.tick_params(labelsize=cfg["tick_size"])
plot_timeseries_dual_axis(ax_ts, water_contents, mod_lat.time.values[0], cfg)
ax_ts.set_yscale("log")
ax_ts.set_ylim((1.0e-1, 1e5))

for f in glob.glob(str(GFX_PNG / "lat_height_frame_*.png")):
    os.remove(f)

if cfg["plot_all_frames"]:
    for iframe in range(IWC_lat.shape[0]):
        update_frame_enhanced(
            iframe, fig, ax_main, ax_ts, pmesh_lh, cfg, mod_lat.time.values, IWC_lat, water_contents
        )
    pat = str(GFX_PNG / f"lat_height_frame_%03d_{cfg['resolution']}.png")
    mp4_lh = GFX_MP4 / f"lat_height_3d_{flare_exp_name}_{cfg['resolution']}_{cs_run.replace('__', '_')}.mp4"
    gif_lh = mp4_lh.with_suffix(".gif")
    tools.convert_to_video(pat, str(mp4_lh), resolution="1920:1080", loop_count=2, framerate=20)
    tools.convert_to_gif(pat, str(gif_lh), scale_factor=0.5, fps=10)
    print(mp4_lh, gif_lh)
else:
    update_frame_enhanced(0, fig, ax_main, ax_ts, pmesh_lh, cfg, mod_lat.time.values, IWC_lat, water_contents)
